In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/home/yuanshanwu/Documents/GitHub/QuantUS/engines/ceus/CLI-Demos
/home/yuanshanwu/Documents/GitHub/QuantUS/engines/ceus


In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['mp4', 'nifti', 'avi']


In [3]:
scan_type = 'nifti'

# Takes the DICOM file as input for contrast enhanced ultrasound (CEUS) scans
CEUS_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_CEUS.nii'
bmode_scan_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/UCSD-P05-V02-CE1_09.45.08_mf_sip_capture_50_2_1_0_BMODE.nii'
scan_loader_kwargs = {
}

In [4]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, CEUS_scan_path, **scan_loader_kwargs)
bmode_image_data = scan_loading_step(scan_type, bmode_scan_path, **scan_loader_kwargs)

In [5]:
seg_type = 'nifti'

seg_path = '/home/yuanshanwu/Documents/TUL/CEUS-Studies/P05_V02_CE01/NewInterpolation/UCSD-P05-V02-CE1_09.45.08/MotionCompensated_QUANTUSGUI.nii.gz'
seg_loader_kwargs = {}

In [6]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, CEUS_scan_path, **seg_loader_kwargs)

# Begin Visualization

In [ ]:
import os
os.environ["QT_API"] = "pyqt6"

import napari
import numpy as np
from napari.qt import thread_worker  # only needed if you want custom playback logic

#
vol_txyz =image_data.pixel_data.transpose(3, 2, 0, 1)
#                                           T  ax sag cor
# Result: (time, ax, sag, cor)

pixdim = image_data.pixdim   # (mm_sag, mm_cor, mm_ax) from NIfTI header
dt     = image_data.frame_rate


viewer = napari.Viewer(title="CEUS 3D Perfusion")

layer = viewer.add_image(
    vol_txyz[:,89:210,:,:],
    name="CEUS Cine",
    scale=(dt, pixdim[2], pixdim[0], pixdim[1]),   # (s, mm_ax, mm_sag, mm_cor)
    colormap="inferno",
    rendering="mip",
    contrast_limits=(0, 255),
)

viewer.dims.axis_labels = ["Time (s)", "Axial (mm)", "Sagittal (mm)", "Coronal (mm)"]
viewer.dims.ndisplay = 3

fps = 1.0 / dt

napari.run()

## Showing mask and CEUS image

In [ ]:
import os
os.environ["QT_API"] = "pyqt6"
import napari
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
Z_START   = 89
Z_END     = 210
n_frames  = image_data.pixel_data.shape[-1]

# ── Precompute 4D motion-compensated mask (T, ax, sag, cor) ──────────────────
print("Precomputing motion-compensated masks...")

# image_data.pixel_data is (sag, cor, ax, T)
sag, cor, ax, _ = image_data.pixel_data.shape

mc_4d = np.zeros((n_frames, ax, sag, cor), dtype=np.uint8)  # (T, ax, sag, cor)

for frame_idx in range(n_frames):
    # apply_to_mask returns (sag, cor, ax)
    mc_xyz = seg_data.motion_compensation.apply_to_mask(
                 seg_data.seg_mask, frame_idx, 0)             # (sag, cor, ax)
    
    mc_ceus = mc_xyz * image_data.pixel_data[:, :, :, frame_idx]  # (sag, cor, ax)
    
    # Transpose to (ax, sag, cor) to match vol_txyz axis order
    mc_4d[frame_idx] = mc_ceus.transpose(2, 0, 1)             # (ax, sag, cor)

print(f"Done. Mask shape: {mc_4d.shape}")

# ── Volume ────────────────────────────────────────────────────────────────────
vol_txyz = image_data.pixel_data.transpose(3, 2, 0, 1)       # (T, ax, sag, cor)

pixdim = image_data.pixdim   # (mm_sag, mm_cor, mm_ax)
dt     = image_data.frame_rate
scale  = (dt, pixdim[2], pixdim[0], pixdim[1])               # (s, mm_ax, mm_sag, mm_cor)

Precomputing motion-compensated masks...
Done. Mask shape: (530, 219, 295, 218)


In [8]:
# ── Launch napari ──────────────────────────────────────────────────────────────
viewer = napari.Viewer(title="CEUS 3D Perfusion")

# CEUS volume — cropped Z range
viewer.add_image(
    mc_4d[:, :, :, :],
    name="CEUS Cine",
    scale=scale,
    colormap="inferno",
    rendering="mip",
    contrast_limits=(0, 255),
)

# Motion-compensated mask — cropped to same Z range
# viewer.add_labels(
#     mc_4d[:, Z_START:Z_END, :, :],
#     name="MC mask",
#     scale=scale,
#     opacity=0.5,
# )

viewer.dims.axis_labels = ["Time (s)", "Axial (mm)", "Sagittal (mm)", "Coronal (mm)"]
viewer.dims.ndisplay = 3

napari.run()

In [10]:
import os
os.environ["QT_API"] = "pyqt6"
import napari
import numpy as np

# ── Shared spatial params (same for both modalities) ─────────────────────────
pixdim = image_data.pixdim        # (mm_sag, mm_cor, mm_ax)
dt_ceus  = image_data.frame_rate
dt_bmode = bmode_image_data.frame_rate

AX_START = 89
AX_END   = 210

# ── CEUS: (sag, cor, ax, time) → (time, ax, sag, cor), sliced axially ────────
ceus_txyz = image_data.pixel_data[:, :, AX_START:AX_END, :].transpose(3, 2, 0, 1)

# ── B-mode: same spatial slice, same transpose ────────────────────────────────
bmode_txyz = bmode_image_data.pixel_data[:, :, AX_START:AX_END, :].transpose(3, 2, 0, 1)

# ── Shared scale & translate (physical coords preserved) ─────────────────────
spatial_scale     = (pixdim[2], pixdim[0], pixdim[1])   # (mm_ax, mm_sag, mm_cor)
ax_offset_mm      = AX_START * pixdim[2]

# ── Viewer ────────────────────────────────────────────────────────────────────
viewer = napari.Viewer(title="CEUS + B-mode 3D Overlay")

# B-mode layer — grey underneath
viewer.add_image(
    bmode_txyz,
    name="B-mode",
    scale=(dt_bmode, *spatial_scale),
    translate=(0, ax_offset_mm, 0, 0),
    colormap="gray",
    rendering="mip",
    contrast_limits=(0, 255),
    opacity=0.5,
)

# CEUS layer — coloured on top
viewer.add_image(
    ceus_txyz,
    name="CEUS Cine",
    scale=(dt_ceus, *spatial_scale),
    translate=(0, ax_offset_mm, 0, 0),
    colormap="inferno",
    rendering="mip",
    contrast_limits=(0, 255),
    opacity=0.7,
    blending="additive",   # additive blending so both layers show through
)

viewer.dims.axis_labels = ["Time (s)", "Axial (mm)", "Sagittal (mm)", "Coronal (mm)"]
viewer.dims.ndisplay = 3


napari.run()